# 03 — Quantum ML Models

**Models:** QSVC (Quantum SVC) · VQC (Variational Quantum Classifier)  
**Framework:** Qiskit 1.x + qiskit-machine-learning 0.9.x  
**Qubits:** 4 (barren plateau safe), linear entanglement  
**Training data:** 400-sample stratified subset (PCA-reduced to 4 dims, normalized to [0, π])

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import yaml
from pathlib import Path

from src.models.quantum import QuantumModelTrainer
from src.features.engineering import load_config, load_splits
from src.evaluation.metrics import evaluate_model, CLASS_NAMES

plt.rcParams.update({
    'font.family': 'serif', 'font.size': 11,
    'axes.labelsize': 12, 'figure.dpi': 150,
})

config = load_config('../config/config.yaml')
CLASS_NAMES = config['labeling']['class_names']
FIGURES_DIR = Path('../results/figures')
MODELS_DIR = Path('../results/models')
print('Imports OK')
print(f'Qiskit version:', end=' ')
import qiskit; print(qiskit.__version__)

## 1. Load Pre-computed PCA Quantum Subset

In [ ]:
splits, quantum = load_splits(config)

X_train_q = quantum['X_train_q']
y_train_q = quantum['y_train_q']
X_test_q  = quantum['X_test_q']
y_test_q  = quantum['y_test_q']

print(f"Quantum train: {X_train_q.shape}")
print(f"Quantum test:  {X_test_q.shape}")
print(f"Feature range: [{X_train_q.min():.3f}, {X_train_q.max():.3f}] (should be [0, π])")
print(f"Classes in train: {np.unique(y_train_q)}")

## 2. PCA Visualization

In [ ]:
pca = quantum['pca']

# Explained variance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(1, len(pca.explained_variance_ratio_) + 1),
            pca.explained_variance_ratio_ * 100,
            color='#673AB7', edgecolor='black', linewidth=0.5)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance (%)')
axes[0].set_title('PCA Explained Variance per Component')

cumulative = np.cumsum(pca.explained_variance_ratio_) * 100
axes[1].plot(range(1, len(cumulative) + 1), cumulative,
             'o-', color='#673AB7', linewidth=2, markersize=8)
axes[1].axhline(y=cumulative[-1], color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance (%)')
axes[1].set_title(f'PCA Cumulative Variance ({cumulative[-1]:.1f}% with 4 components)')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'pca_explained_variance.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# 3D scatter of first 3 PCA components
CLASS_COLORS = {
    0: '#4CAF50', 1: '#FF9800', 2: '#2196F3', 3: '#F44336', 4: '#9C27B0'
}

fig = px.scatter_3d(
    x=X_train_q[:, 0], y=X_train_q[:, 1], z=X_train_q[:, 2],
    color=[CLASS_NAMES[y] for y in y_train_q],
    color_discrete_map={n: c for n, c in zip(CLASS_NAMES, CLASS_COLORS.values())},
    title='Quantum Training Data — First 3 PCA Components (normalized to [0, π])',
    labels={'x': 'PC1', 'y': 'PC2', 'z': 'PC3'},
    opacity=0.7,
)
fig.update_traces(marker=dict(size=4))
fig.update_layout(height=550, legend_title='Climate Condition')
fig.show()

## 3. Circuit Architecture

In [ ]:
trainer = QuantumModelTrainer(config)

# Draw and display circuits
from qiskit.circuit.library import zz_feature_map, efficient_su2

n_qubits = config['quantum_ml']['n_qubits']
feature_map = zz_feature_map(
    feature_dimension=n_qubits,
    reps=config['quantum_ml']['qsvc']['feature_map_reps'],
    entanglement=config['quantum_ml']['qsvc']['entanglement'],
)
ansatz = efficient_su2(
    num_qubits=n_qubits,
    reps=config['quantum_ml']['vqc']['ansatz_reps'],
    entanglement=config['quantum_ml']['vqc']['entanglement'],
)

print(f"Feature Map depth: {feature_map.decompose().depth()}")
print(f"Ansatz depth: {ansatz.decompose().depth()}")
print(f"Ansatz parameters: {ansatz.num_parameters}")

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
feature_map.draw('mpl', ax=axes[0], style='clifford', fold=-1)
axes[0].set_title(f'ZZFeatureMap ({n_qubits} qubits, reps=1, linear entanglement)')
ansatz.draw('mpl', ax=axes[1], style='clifford', fold=-1)
axes[1].set_title(f'EfficientSU2 Ansatz ({n_qubits} qubits, reps=2, linear entanglement, {ansatz.num_parameters} params)')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'quantum_circuits.png', dpi=200, bbox_inches='tight')
plt.show()

## 4. Train QSVC

In [ ]:
%%time
qsvc_result = trainer.train_qsvc(
    X_train_q, y_train_q,
    X_test_q, y_test_q,
)
print(f"Training time: {qsvc_result['training_time']:.1f}s")

In [ ]:
# Evaluate QSVC
qsvc_metrics = evaluate_model(
    predict_fn=lambda X: qsvc_result['model'].predict(X),
    X_test=X_test_q,
    y_test=y_test_q,
    model_name='QSVC',
    class_names=CLASS_NAMES,
)
qsvc_metrics['training_time'] = qsvc_result['training_time']
print(qsvc_metrics['classification_report'])
print(f"Accuracy: {qsvc_metrics['accuracy']:.4f}")
print(f"F1 Macro: {qsvc_metrics['f1_macro']:.4f}")

In [ ]:
# Kernel matrix heatmap
km = qsvc_result['kernel_matrix']
fig = px.imshow(
    km,
    color_continuous_scale='Viridis',
    title=f'QSVC Quantum Kernel Matrix (first {km.shape[0]} training samples)',
    labels={'color': 'Fidelity'},
)
fig.update_layout(height=500, width=520)
fig.show()

## 5. Train VQC

In [ ]:
%%time
vqc_result = trainer.train_vqc(
    X_train_q, y_train_q,
    X_test_q, y_test_q,
)
print(f"Training time: {vqc_result['training_time']:.1f}s")
print(f"Iterations: {len(vqc_result['loss_history'])}")

In [ ]:
# Evaluate VQC
vqc_metrics = evaluate_model(
    predict_fn=lambda X: vqc_result['model'].predict(X),
    X_test=X_test_q,
    y_test=y_test_q,
    model_name='VQC',
    class_names=CLASS_NAMES,
)
vqc_metrics['training_time'] = vqc_result['training_time']
print(vqc_metrics['classification_report'])
print(f"Accuracy: {vqc_metrics['accuracy']:.4f}")
print(f"F1 Macro: {vqc_metrics['f1_macro']:.4f}")

In [ ]:
# VQC loss convergence
loss = vqc_result['loss_history']
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(range(1, len(loss) + 1)), y=loss,
    mode='lines', name='VQC Loss',
    line=dict(color='#9C27B0', width=2)
))
# Smoothed
if len(loss) > 10:
    smoothed = pd.Series(loss).rolling(5, center=True).mean()
    fig.add_trace(go.Scatter(
        x=list(range(1, len(loss) + 1)), y=smoothed,
        mode='lines', name='Smoothed (window=5)',
        line=dict(color='#E91E63', width=2, dash='dash')
    ))

fig.update_layout(
    title=f'VQC Training Convergence (COBYLA, {len(loss)} iterations)',
    xaxis_title='Iteration', yaxis_title='Objective (Cross-Entropy)',
    height=400, legend=dict(x=0.7, y=0.9)
)
fig.show()

## 6. Qubit Scalability Analysis

In [ ]:
# Uses first n columns of X_test_q (6 PCA components needed)
# Re-compute PCA with 6 components for scalability analysis
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler

max_qubits = 6
pca_6 = PCA(n_components=max_qubits, random_state=42)
X_train_pca6 = pca_6.fit_transform(splits['X_train'])
X_test_pca6  = pca_6.transform(splits['X_test'])

# Subsample train to 400
rng = np.random.default_rng(42)
idx = rng.choice(len(splits['y_train']), size=400, replace=False)
X_tr6 = X_train_pca6[idx]
y_tr6 = splits['y_train'][idx]

scaling_trainer = QuantumModelTrainer(config)
qubit_scaling_df = scaling_trainer.qubit_scalability_analysis(
    X_tr6, y_tr6,
    X_test_pca6, splits['y_test'],
    qubit_range=[2, 3, 4, 5, 6],
)
qubit_scaling_df.to_csv('../results/quantum_scaling.csv', index=False)
print(qubit_scaling_df)

In [ ]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['F1 Score vs Qubit Count',
                                    'Training Time vs Qubit Count'])

colors = {'QSVC': '#1976D2', 'VQC': '#7B1FA2'}
for model in qubit_scaling_df['model'].unique():
    sub = qubit_scaling_df[qubit_scaling_df['model'] == model]
    fig.add_trace(
        go.Scatter(x=sub['n_qubits'], y=sub['f1_macro'],
                   name=model, mode='lines+markers',
                   line=dict(color=colors.get(model, 'gray'), width=2),
                   marker=dict(size=8)),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=sub['n_qubits'], y=sub['training_time'],
                   name=model, mode='lines+markers',
                   line=dict(color=colors.get(model, 'gray'), width=2),
                   marker=dict(size=8), showlegend=False),
        row=1, col=2
    )

fig.update_layout(title='Quantum Model Scalability with Qubit Count', height=420)
fig.update_xaxes(title_text='Number of Qubits', row=1, col=1)
fig.update_xaxes(title_text='Number of Qubits', row=1, col=2)
fig.update_yaxes(title_text='F1 Macro', row=1, col=1)
fig.update_yaxes(title_text='Training Time (s)', row=1, col=2)
fig.show()

## 7. Save Results

In [ ]:
import json

trainer.save_models(MODELS_DIR)

quantum_results = {
    'QSVC': {
        'accuracy': qsvc_metrics['accuracy'],
        'f1_macro': qsvc_metrics['f1_macro'],
        'precision_macro': qsvc_metrics['precision_macro'],
        'recall_macro': qsvc_metrics['recall_macro'],
        'training_time': qsvc_result['training_time'],
        'prediction_time': qsvc_metrics['prediction_time'],
        'n_qubits': qsvc_result['n_qubits'],
        'circuit_depth': qsvc_result['circuit_depth'],
    },
    'VQC': {
        'accuracy': vqc_metrics['accuracy'],
        'f1_macro': vqc_metrics['f1_macro'],
        'precision_macro': vqc_metrics['precision_macro'],
        'recall_macro': vqc_metrics['recall_macro'],
        'training_time': vqc_result['training_time'],
        'prediction_time': vqc_metrics['prediction_time'],
        'n_qubits': vqc_result['n_qubits'],
        'n_params': vqc_result['n_params'],
        'circuit_depth': vqc_result['circuit_depth'],
        'n_iterations': len(vqc_result['loss_history']),
    },
}

with open('../results/quantum_results.json', 'w') as f:
    json.dump(quantum_results, f, indent=2)

print('Quantum results saved.')
print(pd.DataFrame(quantum_results).T.to_string())

## 8. (Optional) IBM Quantum Hardware Run

Set `IBM_QUANTUM_TOKEN` env var to enable. Uses ≤20 test samples (~5–10 min).

In [ ]:
import os
if os.environ.get('IBM_QUANTUM_TOKEN'):
    hw_result = trainer.run_on_ibm_hardware(
        X_train_q, y_train_q,
        X_test_q[:20], y_test_q[:20],
    )
    print(f"Hardware backend: {hw_result['backend']}")
    print(f"Hardware vs Simulator agreement: {hw_result.get('hw_sim_agreement', 'N/A'):.1%}")
    print(f"Elapsed: {hw_result['elapsed_s']:.1f}s")
else:
    print("IBM_QUANTUM_TOKEN not set — skipping hardware run.")
    print("Set export IBM_QUANTUM_TOKEN=your_token to enable.")

## Summary

| Model | Accuracy | F1 Macro | Train Time | Qubits | Circuit Depth |
|-------|----------|----------|------------|--------|---------------|
| QSVC  | —        | —        | —          | 4      | —             |
| VQC   | —        | —        | —          | 4      | —             |

*(Values populate after running the cells above)*

**Next:** Phase 4 — Full comparison and visualizations.